# 🎙️ F5 Myanmar TTS Server — JARVIS အတွက် (v2)

**ဒေါ်အေးသန့် အသံနဲ့ မြန်မာလို ပြောတဲ့ TTS server။**
JARVIS (`E:\Mark-LIV-main`) က ဒီ server ကို Cloudflare tunnel ကနေ ခေါ်သုံးပါတယ်။

**အသုံးပြုနည်း:** `Runtime → Change runtime type → T4 GPU` ဖြစ်ကြောင်း စစ်ပြီး **Run all** နှိပ်ပါ။
နောက်ဆုံး (4) cell က ထုတ်ပေးတဲ့ `trycloudflare.com` URL ကို JARVIS ဘက်က
`voice/colab_tts.py` ထဲက `COLAB_URL` မှာ ထည့်ပါ။

---

### v1 နဲ့ ကွာခြားချက်

| | v1 (အရင်) | v2 (အခု) |
|---|---|---|
| Model | `f5-tts` ရဲ့ default `F5TTS_v1_Base` (အင်္ဂလိပ်/တရုတ်သာ) — **မြန်မာစာ လုံးဝ မဖတ်နိုင်** | `f5-myanmar-tts` (မြန်မာ ၇၉၄ နာရီနဲ့ train ထားတဲ့ F5 v2, 24 kHz) |
| Reference အသံ | ကိုယ်တိုင် upload လုပ်ရ (runtime အသစ်တိုင်း ပြန်လုပ်) | **JARVIS ကိုယ်တိုင် တင်ပေးတယ်** — ဘာမှ လုပ်စရာ မလိုတော့ဘူး |
| Error ရှာရင် | server ရဲ့ traceback ကို ဘယ်မှာမှ မမြင်ရ | `/content/server.log` မှာ သိမ်း၊ cell (5) က ပြပေးတယ် |

---

### Server ရဲ့ လမ်းကြောင်းများ (JARVIS သုံးတာ)

- `POST /speak` — `{"text": "…"}` → 24 kHz mono WAV
- `GET  /` — အခြေအနေ (model, reference audio ရှိ/မရှိ)
- `GET/POST /ref` — reference အသံ ဖတ်/အစားထိုး
- `POST /reftext` — reference အသံ ရဲ့ စာသား သတ်မှတ်
- `GET  /log` — server log ရဲ့ နောက်ဆုံးလိုင်းများ (`EXPOSE_LOG = True` ဖြစ်မှ)

> JARVIS က `/speak` ကို ပထမဆုံး ခေါ်တဲ့အခါ model (၆၇၄ MB) ကို download လုပ်ပါတယ် —
> မိနစ်အနည်းငယ် ကြာနိုင်ပါတယ်။ နောက်ပိုင်း စက္ကန့်အနည်းငယ်ပါ။


In [ ]:
# Cell 1: install — မြန်မာ fine-tune လုပ်ထားတဲ့ F5 model + flask
!pip install -q f5-myanmar-tts flask flask-cors 2>&1 | tail -3
print("install done")


In [ ]:
# Cell 2: cloudflared (tunnel) — internet ကနေ ဒီ server ကို ခေါ်နိုင်အောင်
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
print("cloudflared ready")


In [ ]:
# Cell 3: Config — အခြေအနေ ကြည့်ရန် (ပြင်စရာ မလိုပါ)
import os

# ဒေါ်အေးသန့် reference အသံ + အဲဒီအသံ ဖတ်ထားတဲ့ စာသား
# နှစ်ခုလုံးကို JARVIS ကိုယ်တိုင် တင်ပေးပါတယ် (runtime အသစ်တိုင်း အလိုအလျောက်)
REF_WAV_PATH  = "/content/ref_aye_thant.wav"
REF_TEXT_FILE = "/content/ref_text.txt"

# server log ကို tunnel ကနေ ဖတ်ခွင့်ပြုမလား (GET /log) — ပြဿနာရှာရလွယ်အောင်
# မလိုချင်ရင် False လုပ်ပါ
EXPOSE_LOG = True


def show_state():
    ok = os.path.exists(REF_WAV_PATH)
    txt = ""
    if os.path.exists(REF_TEXT_FILE):
        txt = open(REF_TEXT_FILE, encoding="utf-8").read().strip()
    print(f"reference audio : {REF_WAV_PATH} — {'ရှိပါတယ်' if ok else 'မရှိသေးပါ (JARVIS က တင်ပေးပါမယ်)'}")
    print(f"reference text  : {txt or 'မရှိသေးပါ (JARVIS က တင်ပေးပါမယ်)'}")
    if not ok:
        print("                  → မရှိသေးရင် model ရဲ့ ကိုယ်ပိုင် အသံနဲ့ မြန်မာလို ဖတ်ပါမယ်")


show_state()


In [ ]:
# Cell 4: Server + tunnel — JARVIS က ဒီ URL ကို ခေါ်သုံးပါမယ်
import os, re, subprocess, sys, time

# အရင် run ထားတဲ့ server/tunnel ရှိရင် အရင်ပိတ် (Run all ကို နှစ်ခါ run ရင်
# port 5000 မလွတ်လို့ ဒုတိယ server က မတက်နိုင်ပါ)
subprocess.run("pkill -f tts_server.py", shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
time.sleep(1)

SERVER_CODE = r'''
import hashlib
import os
import time

from flask import Flask, request, send_file
from flask_cors import CORS

app = Flask(__name__)
CORS(app)

REF_WAV       = """__REF_WAV__"""
REF_TEXT_FILE = """__REF_TEXT_FILE__"""
OUT_WAV       = "/content/_speak_out.wav"
LOG_PATH      = "/content/server.log"
EXPOSE_LOG    = __EXPOSE_LOG__


# ---- the model: lazy, because the first /speak pays the download ------------
_tts = None


def get_tts():
    global _tts
    if _tts is None:
        from f5_myanmar_tts import MyanmarTTS
        _tts = MyanmarTTS()
    return _tts


def _ref_text():
    if os.path.exists(REF_TEXT_FILE):
        try:
            t = open(REF_TEXT_FILE, encoding="utf-8").read().strip()
            if t:
                return t
        except OSError:
            pass
    return ""


def _ref_state():
    """What the reference is right now, so the client can keep it in sync.

    The clip and its transcript are a pair: the transcript of one recording
    spoken over a different recording makes F5 produce fluent Burmese nonsense,
    which no error message would say. So the client compares the clip it has
    with the one here (sha1) and puts both back when they differ.
    """
    has_wav = os.path.exists(REF_WAV)
    digest = ""
    if has_wav:
        with open(REF_WAV, "rb") as f:
            digest = hashlib.sha1(f.read()).hexdigest()[:12]
    return dict(ref=has_wav, ref_text=_ref_text(), ref_sha1=digest)


@app.route("/speak", methods=["POST"])
def speak():
    data = request.get_json(force=True, silent=True) or {}
    text = (data.get("text") or "").strip()
    if not text:
        return {"error": "empty text"}, 400

    t0 = time.time()
    tts = get_tts()
    ref_text = _ref_text()
    if os.path.exists(REF_WAV) and ref_text:
        tts.speak(text, output_file=OUT_WAV, ref_audio=REF_WAV, ref_text=ref_text)
        used = "cloned reference"
    else:
        # No reference from the client yet: the library's own sample and its own
        # transcript are a matched pair, so the words still come out Burmese —
        # just not in her voice.
        tts.speak(text, output_file=OUT_WAV)
        used = "model's own reference"

    print(f"[speak] {len(text)} chars -> {os.path.getsize(OUT_WAV)} bytes "
          f"in {time.time() - t0:.1f}s ({used})", flush=True)
    return send_file(OUT_WAV, mimetype="audio/wav")


@app.route("/ref", methods=["GET", "POST"])
def ref():
    if request.method == "GET":
        if not os.path.exists(REF_WAV):
            return {"error": "no reference audio on the server yet"}, 404
        return send_file(REF_WAV, mimetype="audio/wav")

    body = request.get_data()
    if len(body) < 4000:
        return {"error": f"too small to be a reference clip ({len(body)} bytes)"}, 400
    with open(REF_WAV, "wb") as f:
        f.write(body)
    print(f"[ref] reference audio replaced: {len(body)} bytes", flush=True)
    return dict(status="ok", bytes=len(body))


@app.route("/reftext", methods=["GET", "POST"])
def reftext():
    if request.method == "GET":
        return {"text": _ref_text()}

    data = request.get_json(force=True, silent=True) or {}
    text = (data.get("text") or "").strip()
    if not text:
        return {"error": "empty text"}, 400
    with open(REF_TEXT_FILE, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"[reftext] reference transcript set ({len(text)} chars)", flush=True)
    return {"status": "ok", "text": text}


@app.route("/log", methods=["GET"])
def log():
    if not EXPOSE_LOG:
        return {"error": "logging is off (EXPOSE_LOG = False in cell 3)"}, 403
    try:
        n = max(1, min(400, int(request.args.get("n", "40"))))
    except (TypeError, ValueError):
        n = 40
    if not os.path.exists(LOG_PATH):
        return {"error": f"{LOG_PATH} does not exist"}, 404
    with open(LOG_PATH, encoding="utf-8", errors="replace") as f:
        lines = f.readlines()
    return {"lines": len(lines), "tail": "".join(lines[-n:])}


@app.route("/", methods=["GET"])
def health():
    return dict(status="ok", service="f5-myanmar-tts",
                model="f5-myanmar-tts v2 (Burmese, 24 kHz)", **_ref_state())


if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000, threaded=True)
'''

code = (SERVER_CODE
        .replace("__REF_WAV__", REF_WAV_PATH)
        .replace("__REF_TEXT_FILE__", REF_TEXT_FILE)
        .replace("__EXPOSE_LOG__", "True" if EXPOSE_LOG else "False"))
with open("/content/tts_server.py", "w", encoding="utf-8") as f:
    f.write(code)
print("server code written -> /content/tts_server.py")

# Start the server with its output going to a LOG FILE rather than a pipe.
# v1 piped it and nothing ever read the pipe, so every traceback vanished into a
# buffer — that is where an afternoon went. A file can be read at any time, by
# cell (5) or by JARVIS over GET /log.
srv_log = open("/content/server.log", "w", encoding="utf-8")
srv = subprocess.Popen([sys.executable, "/content/tts_server.py"],
                       stdout=srv_log, stderr=subprocess.STDOUT, text=True)
time.sleep(5)
print("server started, pid:", srv.pid)

# Tunnel
tun = subprocess.Popen(["/tmp/cloudflared", "tunnel", "--url", "http://127.0.0.1:5000"],
                       stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for _ in range(60):
    line = tun.stdout.readline()
    if not line:
        time.sleep(1)
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print()
if url:
    print("=" * 62)
    print("✅ ဒီ URL ကို JARVIS ဘက်က voice/colab_tts.py → COLAB_URL မှာ ထည့်ပါ")
    print()
    print("   " + url)
    print()
    print("=" * 62)
else:
    print("❌ tunnel URL မရပါ — /content/server.log ကို ကြည့်ပါ")
    for _ in range(20):
        l = tun.stdout.readline()
        if not l:
            break
        print(l, end="")


In [ ]:
# Cell 5: အခြေအနေ စစ်ရန် — ဘယ်အချိန်မဆို ပြန် run လို့ရပါတယ်
import os

print("--- server log (နောက်ဆုံး ၃၀ လိုင်း) ---")
if os.path.exists("/content/server.log"):
    lines = open("/content/server.log", encoding="utf-8", errors="replace").readlines()
    for l in lines[-30:]:
        print(l, end="")
    if not lines:
        print("(လောလောဆယ် ဘာမှ မရှိသေးပါ)")
else:
    print("(log file မရှိသေးပါ — cell 4 ကို အရင် run ပါ)")

print()
print("--- reference အခြေအနေ ---")
wav = os.path.exists("/content/ref_aye_thant.wav")
print("ref audio :", "✅ ရှိပါတယ်" if wav else "❌ မရှိသေးပါ (JARVIS က တင်ပေးပါမယ်)")
if os.path.exists("/content/ref_text.txt"):
    print("ref text  :", open("/content/ref_text.txt", encoding="utf-8").read().strip())
else:
    print("ref text  : ❌ မရှိသေးပါ (JARVIS က တင်ပေးပါမယ်)")
